In [ ]:
# 1. Ambiente e caminhos
import os, json, datetime as dt, numpy as np, pandas as pd, matplotlib.pyplot as plt
try:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = "/content/drive/MyDrive/dissertacao_tinyml"
except ImportError:
    BASE = os.environ.get("BASE_TINYML", "dissertacao_tinyml")
CONJUNTO = os.environ.get("CONJUNTO_TINYML", "coleta")   # "coleta" = dados do nó | "coleta_sim" = sessões sintéticas (notebook 07b)
CONJUNTO = "coleta_sim"
SIM = CONJUNTO != "coleta"
COLETA = f"{BASE}/{CONJUNTO}"; REL = f"{COLETA}/relatorios"; os.makedirs(REL, exist_ok=True)
plt.rcParams.update({"font.family": "serif", "font.serif": ["Times New Roman", "Liberation Serif", "DejaVu Serif"],
                     "font.size": 10, "axes.titlesize": 10.5, "savefig.dpi": 300, "savefig.bbox": "tight"})
virgula = plt.FuncFormatter(lambda v, _: f"{v:g}".replace(".", ",").replace("-", "\u2212"))

In [ ]:
# ==== Verificação de origem das sessões sintéticas — célula única para o Colab ====
# Lê o 07b do Drive, executa as células dele na ordem, com o registro de origem acrescentado,
# gerando em /content/coleta_sim_verif (o coleta_sim/ original NÃO é tocado). Leva de 30 a 60 min.
import json, os
for v in ["ESCALA_TEMPO_TINYML", "LB_HORAS_TINYML", "N_EVENTOS_TINYML"]:
    assert not os.environ.get(v), f"Remova a variável {v} antes de rodar"
NB_07B = "/content/drive/MyDrive/dissertacao_tinyml/Notebooks/07b_gerar_sessoes_sinteticas.ipynb"   # AJUSTE o caminho

CELULA_3B = '# 3b. Registro da origem\nFONTES = []\nCTX = {"sessao": None}\n\ndef _clipe_registrado(self):\n    if not self.fila:\n        self.fila = list(self.rng.permutation(self.membros)); self.usos += 1\n    membro = self.fila.pop()\n    x, fs = sf.read(io.BytesIO(self.z.read(membro)), dtype="float32", always_2d=True)\n    assert fs == FS\n    x = x[:, 0]\n    k = self.rng.integers(len(x))                      # mesmo sorteio do original (início)\n    x = np.roll(x, k)\n    g = self.rng.uniform(-1, 1)                        # mesmo sorteio do original (ganho)\n    FONTES.append({"sessao": CTX["sessao"], "modalidade": "mimii", "banco": getattr(self, "nome", ""),\n                   "id": membro, "deslocamento_amostras": int(k), "ganho_db": round(float(g), 6),\n                   "inicio": "", "fim": ""})\n    return x * 10 ** (g / 20)\n\nBancoAudio._clipe = _clipe_registrado\n\ndef trecho_bristol(grades, horas, rng, tmax=26.0, tmin=18.0):\n    """Idêntica à da célula 3, com registro do dispositivo e do intervalo usados."""\n    n = int(horas * 360)\n    candidatos = []\n    for disp, g in grades.items():\n        for _, seg in g.groupby("trecho"):\n            if len(seg) >= n: candidatos.append((disp, seg))\n    rng.shuffle(candidatos)\n    for folga in (0.0, 1.0, 2.0):\n        for disp, seg in candidatos:\n            for _ in range(20):\n                i = rng.integers(0, len(seg) - n + 1)\n                w = seg.iloc[i:i + n]\n                if w.temp.max() <= tmax + folga * 0.5 and w.temp.min() >= tmin - folga * 2:\n                    FONTES.append({"sessao": CTX["sessao"], "modalidade": "bristol", "banco": "",\n                                   "id": disp, "deslocamento_amostras": "", "ganho_db": "",\n                                   "inicio": str(w.index[0]), "fim": str(w.index[-1])})\n                    return disp, w[["temp", "umid", "press"]].to_numpy(np.float64)\n    raise RuntimeError("Nenhum trecho do Bristol com a duração e a faixa pedidas")\n\n\n'
CELULA_5B = 'for p, b in BANCOS.items():\n    for tipo, banco_ in b.items(): banco_.nome = f"{p}/{tipo}"\n'
CELULA_7 = 'SESS = []\nfor dia, s in enumerate(["SIM001", "SIM002", "SIM003", "SIM004", "SIM005", "SIM006"]):\n    CTX["sessao"] = s\n    print("Gerando", s, "..."); SESS.append(sintetizar(s, dia)); print("  ", SESS[-1])\nS = pd.DataFrame(SESS)\nS[["sessao", "tipo_sessao", "papel", "inicio_real", "offset_termo_s", "fundo_inicio", "fundo_fim"]].to_csv(f"{SAIDA}/sessoes.csv", index=False)\nF = pd.DataFrame(FONTES); F["papel"] = F.sessao.map(P["papel"])\nF.to_csv(f"{SAIDA}/fontes_todas_sessoes.csv", index=False)\nfor s, g in F.groupby("sessao"): g.to_csv(f"{SAIDA}/{s}/fontes.csv", index=False)\nprint("Registros de origem:", len(F))\n\n\n'
CELULA_9 = 'import hashlib, glob\ndef sha256(p, bloco=1 << 20):\n    h = hashlib.sha256()\n    with open(p, "rb") as f:\n        for b in iter(lambda: f.read(bloco), b""): h.update(b)\n    return h.hexdigest()\n\n# (a) As sessões regeneradas são idênticas às usadas na dissertação?\ncomp = []\nfor s in P["papel"]:\n    for orig in sorted(glob.glob(f"{ORIGINAL}/{s}/*")):\n        nome = os.path.basename(orig); novo = f"{SAIDA}/{s}/{nome}"\n        if nome == "fontes.csv": continue\n        comp.append({"sessao": s, "arquivo": nome, "identico": os.path.exists(novo) and sha256(orig) == sha256(novo)})\nfor nome in ["planilha_eventos.csv", "sessoes.csv"]:\n    comp.append({"sessao": "-", "arquivo": nome, "identico": sha256(f"{ORIGINAL}/{nome}") == sha256(f"{SAIDA}/{nome}")})\nC = pd.DataFrame(comp); C.to_csv(f"{SAIDA}/comparacao_hashes.csv", index=False)\nprint("Arquivos comparados:", len(C), "| idênticos:", int(C.identico.sum()))\nif not C.identico.all():\n    print("ATENÇÃO — arquivos diferentes (NÃO substituir nada em coleta_sim):")\n    print(C[~C.identico].to_string(index=False))\n\n# (b) MIMII: cada gravação pertence a uma única partição, e cada sessão usou só a sua\npart_unica = inv.groupby("membro").particao.nunique()\nmi = F[F.modalidade == "mimii"].merge(inv[["membro", "particao"]].drop_duplicates(), left_on="id", right_on="membro", how="left")\nprint("Gravações do MIMII em mais de uma partição (inventário):", int((part_unica > 1).sum()))\nprint("Clipes usados fora da partição da sessão:", int((mi.particao != mi.papel).sum()))\nprint("Gravações usadas em sessões de papéis diferentes:", int((mi.groupby("id").papel.nunique() > 1).sum()))\n\n# (c) Bristol: cada trecho está dentro do bloco cronológico do papel, e não há sobreposição entre papéis\nbr = F[F.modalidade == "bristol"].copy()\nbr["ini"] = pd.to_datetime(br.inicio); br["fim_"] = pd.to_datetime(br.fim)\nlim = {"treino": (None, c1), "validacao": (c1, c2), "teste": (c2, None)}\nfora = [r.sessao for r in br.itertuples() if (lim[r.papel][0] is not None and r.ini < lim[r.papel][0]) or\n        (lim[r.papel][1] is not None and r.fim_ >= lim[r.papel][1])]\nsobrep = [(a.sessao, b.sessao) for a in br.itertuples() for b in br.itertuples()\n          if a.sessao < b.sessao and a.id == b.id and a.papel != b.papel and a.ini <= b.fim_ and b.ini <= a.fim_]\nprint("Trechos do Bristol fora do bloco do papel:", fora or "nenhum")\nprint("Sobreposição de trechos Bristol entre papéis:", sobrep or "nenhuma")\nprint(br[["sessao", "papel", "id", "inicio", "fim"]].to_string(index=False))\n\n# (d) Repetição de clipes DENTRO de cada sessão (limitação a declarar)\nrep = mi.groupby(["sessao", "banco"]).agg(usos=("id", "size"), clipes_distintos=("id", "nunique"))\nrep["usos_por_clipe"] = (rep.usos / rep.clipes_distintos).round(1)\nprint(rep.to_string())\nrep.to_csv(f"{SAIDA}/repeticao_clipes.csv")\n\n# Copie para o Drive apenas os registros (não os WAV):\nDEST = f"{BASE}/manifesto/origem"; os.makedirs(DEST, exist_ok=True)\nfor arq in ["fontes_todas_sessoes.csv", "comparacao_hashes.csv", "repeticao_clipes.csv"]:\n    shutil.copy(f"{SAIDA}/{arq}", DEST)\nprint("Registros copiados para", DEST)\n'

nb = json.load(open(NB_07B, encoding="utf-8"))
codigo = ["".join(c["source"]) for c in nb["cells"] if c["cell_type"] == "code"]
assert len(codigo) == 8, f"esperadas 8 células de código no 07b, encontradas {len(codigo)}"
G = globals()
for i, cel in enumerate(codigo, start=1):
    if i == 1:   # célula 1: saída em disco local e referência ao original
        cel = cel.replace('SAIDA = f"{BASE}/coleta_sim"; os.makedirs(SAIDA, exist_ok=True)',
                          'ORIGINAL = f"{BASE}/coleta_sim"\nSAIDA = f"{LOCAL}/coleta_sim_verif"; os.makedirs(SAIDA, exist_ok=True)')
        assert "coleta_sim_verif" in cel, "não encontrei a linha do SAIDA na célula 1"
    if i == 7:   # célula 7 (geração das seis sessões) é substituída pela versão com registro
        cel = CELULA_7
    print(f"--- executando célula {i} ---")
    exec(cel, G)
    if i == 3:   # depois das funções do gerador
        exec(CELULA_3B, G)
    if i == 5:   # depois dos bancos do MIMII e dos blocos do Bristol
        exec(CELULA_5B, G)
print("--- verificações ---")
exec(CELULA_9, G)